## Text Classification

In [1]:
# Text classification is a common NLP task that assigns a label or class to text.
# One of the most popular forms of text classification is sentiment analysis, which assigns a label like 🙂 positive, 🙁 negative, or 😐 neutral to a sequence of text.

In [2]:
""" 

1. Finetune DistilBERT [https://huggingface.co/distilbert/distilbert-base-uncased] on the IMDB dataset [https://huggingface.co/imdb/datasets] to determine whether a movie
review is positive or negative.
2. Use your finetuned model for inference
"""

' \n\n1. Finetune DistilBERT [https://huggingface.co/distilbert/distilbert-base-uncased] on the IMDB dataset [https://huggingface.co/imdb/datasets] to determine whether a movie\nreview is positive or negative.\n2. Use your finetuned model for inference\n'

In [4]:
!pip install transformers datasets evaluate accelerate scikit-learn

  Using cached scikit_learn-1.8.0-cp311-cp311-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.8.0-cp311-cp311-macosx_12_0_arm64.whl (8.1 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 28.3 MB/s  0:00:00m0:00:0100:01
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [scikit-learn] [scikit-learn]


In [5]:
from huggingface_hub import notebook_login

notebook_login()

## Load IMDB dataset

In [6]:
from datasets import load_dataset

imdb = load_dataset("imdb")

In [8]:
imdb.shape

{'train': (25000, 2), 'test': (25000, 2), 'unsupervised': (50000, 2)}

In [9]:
imdb["test"][0]

# 2 fields: one is the text and another is the label
# text: movie review
# label: 0 for a negative review and 1 for a positive review

{'text': 'I love sci-fi and am willing to put up with a lot. Sci-fi movies/TV are usually underfunded, under-appreciated and misunderstood. I tried to like this, I really did, but it is to good TV sci-fi as Babylon 5 is to Star Trek (the original). Silly prosthetics, cheap cardboard sets, stilted dialogues, CG that doesn\'t match the background, and painfully one-dimensional characters cannot be overcome with a \'sci-fi\' setting. (I\'m sure there are those of you out there who think Babylon 5 is good sci-fi TV. It\'s not. It\'s clichéd and uninspiring.) While US viewers might like emotion and character development, sci-fi is a genre that does not take itself seriously (cf. Star Trek). It may treat important issues, yet not as a serious philosophy. It\'s really difficult to care about the characters here as they are not simply foolish, just missing a spark of life. Their actions and reactions are wooden and predictable, often painful to watch. The makers of Earth KNOW it\'s rubbish as 

## Preprocess

In [10]:
# load a DistilBERT tokenizer to preprocess the text field

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

""" 
1. Converts raw text -> tokens (numbers the model understands)
2. Applies preprocessing like:
    (a) Lowercasing text
    (b) Splitting words into subwords
3. Maps words/subwords to IDs using a vocabulary
"""

' \n1. Converts raw text -> tokens (numbers the model understands)\n2. Applies preprocessing like:\n    (a) Lowercasing text\n    (b) Splitting words into subwords\n3. Maps words/subwords to IDs using a vocabulary\n'

In [11]:
# preprocessing function to tokenize text and truncate sequences and truncate sequences to be no longer than DistilBERT's maximum input length.

def preprocess_function(examples):
    return tokenizer(examples["text"], truncation = True)

In [12]:
# to apply preprocess function over the entire dataset, use map function

tokenized_imdb = imdb.map(preprocess_function, batched = True) # to process multiple elements of the dataset at once

In [13]:
# now create a batch of examples using DataCollatorWithPadding
# more efficient to dynamically pad the sentences to the longest length in a batch during collation, instead of padding the whole dataset to the maximum length

from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## Evaluate

In [14]:
# including a metric during training is often helpful for evaluating a model's performance

import evaluate

accuracy = evaluate.load("accuracy")

W0405 10:46:11.440000 89200 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [15]:
import numpy as np


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis = 1)
    return accuracy.compute(predictions=predictions, references=labels)

## Train


In [17]:
id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}

In [18]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer


model = AutoModelForSequenceClassification.from_pretrained("distilbert/distilbert-base-uncased", 
                                                           num_labels = 2,
                                                           id2label = id2label,
                                                           label2id = label2id)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [19]:
""" 
1. Define your training hyperparameters in TrainingArguments. At the end of each epoch, the Trainer will evaluate the accuracy and save the training checkpoint.
2. Pass the training arguments to Trainer alongside with the model, dataset, tokenizer, data collator, and compute_metrics function
3. Call train() to finetune your model.
"""

' \n1. Define your training hyperparameters in TrainingArguments. At the end of each epoch, the Trainer will evaluate the accuracy and save the training checkpoint.\n2. Pass the training arguments to Trainer alongside with the model, dataset, tokenizer, data collator, and compute_metrics function\n3. Call train() to finetune your model.\n'

In [23]:
training_args = TrainingArguments(
    output_dir="my_awesome_text_classification_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=True
)

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset=tokenized_imdb["train"],
    eval_dataset = tokenized_imdb["test"],
    tokenizer= tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

  0%|          | 0/3126 [00:00<?, ?it/s]

/Users/koushalsmodi/Desktop/MachineLearning/MachineLearningProjects/NLP/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


{'loss': 0.3187, 'grad_norm': 13.25955581665039, 'learning_rate': 1.6801023672424827e-05, 'epoch': 0.32}
{'loss': 0.2446, 'grad_norm': 7.047168731689453, 'learning_rate': 1.3602047344849649e-05, 'epoch': 0.64}


RuntimeError: MPS backend out of memory (MPS allocated: 20.02 GiB, other allocations: 7.02 GiB, max allowed: 27.20 GiB). Tried to allocate 192.00 MiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

In [24]:
trainer.push_to_hub()

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/koushalsmodi/my_awesome_text_classification_model/commit/2ea3f9fa91c9465c010b651e969345ff3e3f5d04', commit_message='End of training', commit_description='', oid='2ea3f9fa91c9465c010b651e969345ff3e3f5d04', pr_url=None, repo_url=RepoUrl('https://huggingface.co/koushalsmodi/my_awesome_text_classification_model', endpoint='https://huggingface.co', repo_type='model', repo_id='koushalsmodi/my_awesome_text_classification_model'), pr_revision=None, pr_num=None)

## Inference

In [25]:
text1 = "This was a masterpiece. Not completely faithful to the books, but enthralling from beginning to end. Might be my favorite of the three."
text2 = "This wasn't a masterpiece. Quite disappointing."

In [26]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis", model = "koushalsmodi/my_awesome_text_classification_model")
classifier(text1)

config.json:   0%|          | 0.00/746 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

[{'label': 'POSITIVE', 'score': 0.994264543056488}]

In [27]:
classifier(text2)

[{'label': 'NEGATIVE', 'score': 0.9678788781166077}]